E-Commerce Sales - Python Cleaning Skripti


In [1]:
#Bu skript dataseti Power BI-a göndərməzdən əvvəl
#bütün lazımi əməliyyatları yerinə yetirir.

In [2]:
import pandas as pd
import numpy as np
import datetime as dt

In [3]:
df = pd.read_csv("data - data.csv.csv", 
                 encoding = "latin1", #xususi simvollar ucun
                 dtype = {"CustomerID": str,
        "StockCode": str})
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/2011 12:50,0.85,12680,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680,France


In [4]:

print(f"      {len(df):,} sətir, {len(df.columns)} sütun yükləndi, ")

      541,909 sətir, 8 sütun yükləndi, 


Sutun adlarinin temizlenmesi

In [5]:
df.columns = (
      df.columns
      .str.lower())

In [6]:
print (f" sutun adlari kicik herfle yazildi")
print(f"      Sütunlar: {list(df.columns)}")

 sutun adlari kicik herfle yazildi
      Sütunlar: ['invoiceno', 'stockcode', 'description', 'quantity', 'invoicedate', 'unitprice', 'customerid', 'country']


Tarix sutunlarinin temizlenmesi

In [7]:
df["invoicedate"] = pd.to_datetime(df["invoicedate"])


In [8]:
print(f"tarix araligi: { df["invoicedate"].min().date()} -> {df["invoicedate"].max().date()}")

tarix araligi: 2010-12-01 -> 2011-12-09


Null deyerlerin yoxlanmasi

In [9]:
 df.isnull().sum()


invoiceno           0
stockcode           0
description      1454
quantity            0
invoicedate         0
unitprice           0
customerid     135080
country             0
dtype: int64

In [10]:
df['customerid'] = df['customerid'].fillna('unknown') #datasetde 27% teskil etdiyinden, musteri analizinde ciddi data itkisi ola bilerdi
df = df.dropna(subset = ['description']) # kicik bir hisse oldugundan siline biler

Dublikatlarin  idare olunmasi


In [11]:
dublikatlarin_sayi= df.duplicated().sum()

In [12]:
print(f' dublikat ssetirlerin sayi: {dublikatlarin_sayi} ')

 dublikat ssetirlerin sayi: 5268 


In [13]:
df = df.drop_duplicates() #tarix sutunundaki deyisiklikden once yeniden yoxlanildi, eyni saatda da oldugundan silindi

Yeni sutunlarin elave olunmasi

In [14]:
df

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France


In [15]:


# Movsum teyin eden funksiya
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

df['season'] = df['invoicedate'].dt.month.apply(get_season)


In [16]:
df['revenue'] = df['unitprice']* df['quantity']

In [17]:
df['stockcode'].nunique()

3958

In [18]:
df.groupby('customerid')['revenue'].sum()

customerid
12346            0.00
12347         4310.00
12348         1797.24
12349         1757.55
12350          334.40
              ...    
18281           80.82
18282          176.60
18283         2045.53
18287         1837.28
unknown    1447487.53
Name: revenue, Length: 4373, dtype: float64

In [19]:
df = df[df['revenue'] > 0] #revenue-su 0dan boyuk olmayanlar datasetden cixarildi


In [20]:
df

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,season,revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,Winter,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Winter,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,Winter,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Winter,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Winter,20.34
...,...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680,France,Winter,10.20
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,Winter,12.60
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,Winter,16.60
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,Winter,16.60


In [21]:
df.groupby('customerid')['revenue'].sum().sort_values(ascending= False).head(10) #en cox alis-veris eden musteriler

customerid
unknown    1754901.91
14646       280206.02
18102       259657.30
17450       194390.79
16446       168472.50
14911       143711.17
12415       124914.53
14156       117210.08
17511        91062.38
16029        80850.84
Name: revenue, dtype: float64

In [22]:
df.groupby("description")["revenue"].sum().sort_values(ascending = False).head(10) #en cox gelir elde oluanan mehsullar

description
DOTCOM POSTAGE                        206248.77
REGENCY CAKESTAND 3 TIER              174156.54
PAPER CRAFT , LITTLE BIRDIE           168469.60
WHITE HANGING HEART T-LIGHT HOLDER    106236.72
PARTY BUNTING                          99445.23
JUMBO BAG RED RETROSPOT                94159.81
MEDIUM CERAMIC TOP STORAGE JAR         81700.92
POSTAGE                                78101.88
Manual                                 77752.82
RABBIT NIGHT LIGHT                     66870.03
Name: revenue, dtype: float64

In [23]:
df.groupby("description")["revenue"].sum().sort_values(ascending = True).head(10) #en az gelir elde oluanan mehsullar

description
PADS TO MATCH ALL CUSHIONS             0.003
HEN HOUSE W CHICK IN NEST              0.420
SET 12 COLOURING PENCILS DOILEY        0.650
VINTAGE BLUE TINSEL REEL               0.840
PINK CRYSTAL GUITAR PHONE CHARM        0.850
HAPPY BIRTHDAY CARD TEDDY/CAKE         0.950
CAT WITH SUNGLASSES BLANK CARD         0.950
60 GOLD AND SILVER FAIRY CAKE CASES    1.100
SET 36 COLOURING PENCILS DOILEY        1.250
ORANGE FELT VASE + FLOWERS             1.250
Name: revenue, dtype: float64

In [24]:
df.groupby('country')['revenue'].sum().sort_values(ascending = False).head() #en cox gelir elde olunan olkeler

country
United Kingdom    9001744.094
Netherlands        285446.340
EIRE               283140.520
Germany            228678.400
France             209625.370
Name: revenue, dtype: float64

RFM analizi

In [25]:
analysis_date = df['invoicedate'].max() + dt.timedelta(days=1) #Max tarixden bir gun sonrani analiz tarixi olaraqa teyin etdik
rfm = df.groupby('customerid').agg(
    recency=('invoicedate', lambda x: (analysis_date - x.max()).days),
    frequency=('invoicedate', 'count'),
    monetary=('revenue', 'sum')
)




In [26]:
rfm['r_score'] = pd.qcut(rfm['recency'], 5, labels=[5,4,3,2,1])
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['m_score'] = pd.qcut(rfm['monetary'], 5, labels=[1,2,3,4,5])

rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)

In [27]:
def segment(x):
    if x == '555':
        return 'Best Customers'
    elif x[0] == '5':
        return 'Recent Customers'
    elif x[1] == '5':
        return 'Frequent Buyers'
    elif x[2] == '5':
        return 'Big Spenders'
    else:
        return 'At Risk'

rfm['segment'] = rfm['rfm_score'].apply(segment)


In [28]:
df

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,season,revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,Winter,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Winter,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,Winter,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Winter,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Winter,20.34
...,...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680,France,Winter,10.20
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,Winter,12.60
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,Winter,16.60
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,Winter,16.60


In [29]:
rfm

,recency,frequency,monetary,r_score,f_score,m_score,rfm_score,segment
customerid,,,,,,,,
12346,326,1,77183.60,1,1,5,115,Big Spenders
12347,2,182,4310.00,5,5,5,555,Best Customers
12348,75,31,1797.24,2,3,4,234,At Risk
12349,19,73,1757.55,4,4,4,444,At Risk
12350,310,17,334.40,1,2,2,122,At Risk
...,...,...,...,...,...,...,...,...
18281,181,7,80.82,1,1,1,111,At Risk
18282,8,12,178.05,5,1,1,511,Recent Customers
18283,4,721,2045.53,5,5,4,554,Recent Customers


In [30]:
# seqment sutununu orginal df-e merge edirik
df = df.merge(rfm[['segment']], on='customerid', how='left')




In [31]:
df

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,season,revenue,segment
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,Winter,15.30,Frequent Buyers
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Winter,20.34,Frequent Buyers
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,Winter,22.00,Frequent Buyers
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Winter,20.34,Frequent Buyers
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Winter,20.34,Frequent Buyers
...,...,...,...,...,...,...,...,...,...,...,...
524873,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680,France,Winter,10.20,Recent Customers
524874,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,Winter,12.60,Recent Customers
524875,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,Winter,16.60,Recent Customers
524876,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,Winter,16.60,Recent Customers


In [32]:
# dataseti csv halina salib vizuallasdirmaya hazir hala getiririk
df.to_csv("e_commerce.last.csv", index=False)


In [57]:
at_risk_customers = df[df['segment'] == 'At Risk']

# her risk grupundaki alici ucun son tarix
last_purchase = at_risk_customers.groupby('customerid')['invoicedate'].max()
average_last_purchase = last_purchase.mean()
average_last_purchase 

Timestamp('2011-08-04 23:16:30.349699584')